# Solution C2 · Adding copies of a feature makes it look useless

**Type**: Break the model

## Objective

Add `k` almost identical copies of the most important feature and observe that the permutation importance of **each** copy drops toward zero (the model can use the others) while the sum stays and the impurity importance is split. Then fix it with the **group permutation**:
shuffling the correlated columns together.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

X, y = make_classification(2000, n_features=8, n_informative=4, n_redundant=0, class_sep=1.0, flip_y=0.05, shuffle=False, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
rf = lambda: RandomForestClassifier(200, random_state=0, n_jobs=-1)


def with_copies(k, seed=0):
    rng = np.random.default_rng(seed)
    copies_tr = [X_tr[:, 0] + rng.normal(0, 0.05, len(X_tr)) for _ in range(k)]
    copies_te = [X_te[:, 0] + rng.normal(0, 0.05, len(X_te)) for _ in range(k)]
    return np.column_stack([X_tr] + copies_tr), np.column_stack([X_te] + copies_te)

## Commented solution

In [ ]:
f0 = rf().fit(X_tr, y_tr)
perm0 = permutation_importance(f0, X_te, y_te, n_repeats=10, random_state=0).importances_mean[0]
print(f"permutation importance of column 0 without copies: {perm0:.3f}")

In [ ]:
table = pd.DataFrame(index=[1, 3, 5], columns=["acc", "orig_perm", "mean_single_copy_perm", "orig_imp", "group_imp"], dtype=float)
models = {}
for k in table.index:
    A_tr, A_te = with_copies(k)
    f = rf().fit(A_tr, y_tr)
    models[k] = (f, A_te)
    perm = permutation_importance(f, A_te, y_te, n_repeats=10, random_state=0).importances_mean
    group = [0] + list(range(8, 8 + k))
    table.loc[k] = [f.score(A_te, y_te), perm[0], perm[8:8 + k].mean(), f.feature_importances_[0], f.feature_importances_[group].sum()]
table.round(3)

In [ ]:
def group_perm(model, X, y, columns, n_repeats=10, seed=0):
    rng = np.random.default_rng(seed)
    base = model.score(X, y)
    losses = []
    for _ in range(n_repeats):
        p = rng.permutation(len(X))
        Xp = X.copy()
        Xp[:, columns] = X[p][:, columns]                       # same permutation for all the columns of the group
        losses.append(base - model.score(Xp, y))
    return float(np.mean(losses))


f5, A_te5 = models[5]
group_perm_k5 = group_perm(f5, A_te5, y_te, [0] + list(range(8, 13)))
orig_perm_at_k5_is_zero = bool(table.loc[5, "orig_perm"] < 0.02)
group_recovers = bool(group_perm_k5 > 0.8 * perm0)
print(f"group permutation (column 0 and 5 copies) = {group_perm_k5:.3f} (without copies: {perm0:.3f}) | column 0 alone with 5 copies: {table.loc[5, 'orig_perm']:.3f}")

## Check

In [ ]:
with verify():
    check(perm0 is not None and 0.15 < perm0 < 0.25, "without copies column 0 is worth about 0.2 of lost accuracy (0.15-0.25)", "permutation_importance(...).importances_mean[0]")
    check(table.notna().all().all() and table.loc[1, "orig_perm"] < 0.7 * perm0 and table.loc[3, "orig_perm"] < 0.1 * perm0 + 0.01, "with 1 copy the permutation of column 0 drops below 70%; with 3 below 15%", "the importance is split among the copies")
    check(table.loc[[3, 5], "orig_perm"].max() < 0.02 and table.loc[[3, 5], "mean_single_copy_perm"].max() < 0.02, "with 3 and 5 copies each column, alone, is worth less than 0.02: it looks useless", "the model uses the other copies")
    check(table["orig_imp"].is_monotonic_decreasing and table["group_imp"].is_monotonic_increasing, "the impurity of column 0 drops, that of the group rises: more copies = more total weight", "sum of the importances of the group")
    check(group_perm_k5 is not None and group_perm_k5 > 0.8 * perm0 and orig_perm_at_k5_is_zero is True and group_recovers is True, "permuting the group together recovers the importance (over 80% of the initial one)", "same row permutation for all the columns of the group")
done()

## What you learned

- **With one copy the permutation importance halves, with three or five it disappears:** 0.199 without copies → 0.103 with one copy → about 0 with three (−0.001) and with five (0.009). Shuffling a single column does not take information away from the model, which finds it in the copies. A report listing the individual importances would conclude that "x0 does not matter": it is the most important variable of the problem.
- **The impurity importance is split** (0.21 → 0.05 for the original column) but the **sum of the group rises** (0.34 with five copies): the forest uses the copies instead of the original, precisely because they are interchangeable.
- **The group permutation** (shuffling the correlated columns together, with the same row permutation) measures the importance of the **information** and not of the single column: it recovers (and exceeds) the original value: 0.24 against 0.2. It applies to groups you have identified by correlation or meaning (the measurements of the same sensor, the variables of a single record).
- What to do in practice: before interpreting the importances **group the correlated features** (correlation matrix, hierarchical clustering) and compute the importance of the groups; or reduce the redundancy (PCA on a group, M6).

## Common pitfall

Discarding the "unimportant" features after a permutation or after the impurity importance without checking the correlations. If you remove them one at a time the copies take their place, but if you remove the whole group you lose the information.

## Interview question

<details><summary>How do you interpret importances with highly correlated variables?</summary>

With caution: the importance (by permutation or from impurity) is split among redundant variables and each looks unimportant. I group the correlated features (correlations, clustering) and compute the group permutation, shuffling all the columns of the group together, or I reduce the redundancy upstream.
I do not remove features based on the individual importance without this check.

</details>